In [ ]:
import os
import torch
import torch.nn as nn
from torch.autograd import Variable
import torch.utils.data as Data
import torchvision
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

learning_rate = 1e-4
keep_prob_rate = 0.7 #
max_epoch = 3
BATCH_SIZE = 50

DOWNLOAD_MNIST = False
if not(os.path.exists('./mnist/')) or not os.listdir('./mnist/'):
    # not mnist dir or mnist is empyt dir
    DOWNLOAD_MNIST = True

train_data = torchvision.datasets.MNIST(root='./mnist/', train=True, transform=torchvision.transforms.ToTensor(), download=DOWNLOAD_MNIST)
train_loader = Data.DataLoader(dataset = train_data ,batch_size= BATCH_SIZE ,shuffle= True)

test_data = torchvision.datasets.MNIST(root='./mnist/', train=False, download=DOWNLOAD_MNIST)
# torchvision>=0.3 uses .data/.targets; volatile Variable is removed.
test_x = test_data.data[:500].unsqueeze(1).float() / 255.
test_y = test_data.targets[:500].numpy()

class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(
                # patch 7 * 7 ; 1  in channels ; 32 out channels ; ; stride is 1
                # padding style is same(that means the convolution opration's input and output have the same size)
                in_channels=1,
                out_channels=32,
                kernel_size=7,
                stride=1,
                padding=3,
            ),
            nn.ReLU(),        # activation function
            nn.MaxPool2d(2),  # pooling operation
        )
        self.conv2 = nn.Sequential(
            # line 1 : convolution function, patch 5*5 , 32 in channels ;64 out channels; padding style is same; stride is 1
            # line 2 : choosing your activation funciont
            # line 3 : pooling operation function.
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=5,
                stride=1,
                padding=2,
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),

        )
        self.out1 = nn.Linear( 7*7*64 , 1024 , bias= True)   # full connection layer one

        # nn.Dropout(p=...) takes drop probability (not keep probability).
        self.dropout = nn.Dropout(p=1 - keep_prob_rate)
        self.out2 = nn.Linear(1024,10,bias=True)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = x.view(x.size(0), -1)  # flatten to (batch_size, 7 * 7 * 64)
        out1 = self.out1(x)
        out1 = F.relu(out1)
        out1 = self.dropout(out1)
        out2 = self.out2(out1)
        return out2

def test(cnn):
    global prediction
    cnn.eval()
    with torch.no_grad():
        y_pre = cnn(test_x)
        _, pre_index = torch.max(y_pre, 1)
        pre_index = pre_index.view(-1)
        prediction = pre_index.cpu().numpy()
    cnn.train()
    correct = np.sum(prediction == test_y)
    return correct / 500.0

def train(cnn):
    optimizer = torch.optim.Adam(cnn.parameters(), lr=learning_rate )
    loss_func = nn.CrossEntropyLoss()
    
    history_loss = []
    history_acc = []
    steps = []
    global_step = 0
    
    for epoch in range(max_epoch):
        for step, (x_, y_) in enumerate(train_loader):
            x ,y= Variable(x_),Variable(y_)
            output = cnn(x)  
            loss = loss_func(output,y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            global_step += 1
            
            # 使用列表保存指标用于绘图
            if step != 0 and step % 20 == 0:
                acc = test(cnn)
                history_loss.append(loss.item())
                history_acc.append(acc)
                steps.append(global_step)
                print(f"Epoch {epoch} | Step {step} => Loss: {loss.item():.4f}, Test Acc: {acc:.4f}")
                
    # 绘制曲线
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(steps, history_loss, label='Train Loss', color='red')
    plt.title('PyTorch CNN: Loss (Batch)')
    plt.xlabel('Global Steps')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(steps, history_acc, label='Test Accuracy', color='blue')
    plt.title('PyTorch CNN: Test Accuracy')
    plt.xlabel('Global Steps')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.savefig('pytorch_cnn_curve.png') # 保存图表总结用于Word
    plt.show()

if __name__ == '__main__':
    cnn = CNN()
    train(cnn)

### 框架对比与实验结果总结 (PyTorch)

1. **核心逻辑**：与 TF1.x 不同，这套代码使用的 PyTorch 最大的特点是**动态计算图 (Dynamic Computation Graph)**，也就是被称为 "Define-by-Run" 的技术设计。
2. **代码风格对比**：相比于 TF1 写满各种占位符和推迟执行机制，PyTorch 代码在书写和调试上显得非常直观、Pythonic。我们能直接继承 `nn.Module`，并在 `forward(self, x)` 中自然地跑流程计算。这里所用的 `Variable` (`tensor`) 既装载数据也负责保留它的求导状态。每次通过 `loss.backward()`，它就是在我们能看见的时间点原位进行了自动微分。这种机制让我们能够在训练循环内部随心所欲地打印变量。
3. **可视化表现**：通过增加的可视化代码，我们运行了 3 个 Epoch。可以看到 `Train Loss` 随着迭代步数的增加迅速产生有效收敛。测试阶段，在经历很短的几十步之后，`Test Accuracy` 测试的分类准确率就会呈现稳定的对数增长状态。
4. **结论**：虽然搭建两者的 CNN（网络层配置：2个卷积池化层块，接入全连接分类头）结构基本上是一致的，所得到的训练效果与特征提取能力也并无太大偏差，但显然 PyTorch 所用的前向后向控制方式更容易理解修改，尤其在进行复杂的模型实验或改动网络参数层时更有优势。